# Subpopulation Analysis

Unified pipeline for selecting neuron subpopulations and inspecting both their
**decoding manifold** and **encoding manifold**.

**Index space convention:**
- *Full space* (0..N-1): rows of `tensor4d`; used for dynamic metrics
- *Nonoutlier space* (0..M-1): rows of `tensor4d_nonout`; `subpop_indices` live here

`tensor4d_nonout = tensor4d[nonoutliers]` is created in Section 1 (after IAN).
All selection methods produce `subpop_mask` and `subpop_indices` in nonoutlier space:
- Method 1: `subpop_mask = selected_mask[nonoutliers]` remaps from full → nonoutlier space
- Methods 2a–2c: operate directly on `cluster_labels`/`embedding_` which are already nonoutlier-space

Decoding analysis then uses:
```python
tensor4d_sub = tensor4d_nonout[subpop_indices]
```

## Table of Contents
0. [Imports & Config](#section-0)
1. [Load Data](#section-1)
2. [Choose Subpopulation](#section-2)
3. [Decoding Analysis](#section-3)
4. [Encoding Analysis](#section-4)
5. [PSTH Heatmaps](#section-5)
6. [Population-Fraction Sweep](#section-6)

## Section 0: Imports & Config <a id='section-0'></a>

**Edit the config cell below before running the notebook.**

In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt
import tueplots
from tueplots import bundles
from tueplots.constants.color import rgb
from sklearn.decomposition import PCA
from scipy.sparse import load_npz, csr_matrix
from scipy.spatial.distance import squareform, pdist

import sys, os; _d = os.path.abspath(os.getcwd()); sys.path.insert(0, _d if os.path.isdir(os.path.join(_d, 'src')) else os.path.dirname(_d))
from src import (
    process_tensor_data, loadPreComputedCP, getPermutedTensor,
    getNeuralMatrix, compute_mds_embedding, run_hdbscan_clustering,
    handle_disconnected_points,
)
from src.subpop_utils import (
    # Group A
    compute_dynamic_metrics, filter_neurons_by_metric,
    select_neurons_by_cluster, select_neurons_by_bbox, select_neurons_by_radius,
    select_top_k_by_metric,
    # Group B
    compute_decoding_manifold, compute_decoding_trajectories,
    knn_decoding_accuracy, procrustes_r2,
    # Group C
    build_encoding_manifold_for_subpop,
    # Group D
    plot_metric_distributions, plot_neuron_locations_in_manifold,
    plot_decoding_manifold, plot_decoding_trajectories,
    plot_encoding_manifold_3d, plot_encoding_manifold_2d,
)

from ian.ian import IAN
from ian.utils import pwdists
from ian.embed_utils import diffusionMapSparseK

logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
plt.rcParams.update(bundles.neurips2021(usetex=False))

In [ ]:
# ============================================================
# USER CONFIG — only change the PREFIX line
# R, method, and low_sf are set automatically from the lookup table below.
# ============================================================

#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_inputs0_maxFr_maxNr_seed1"
#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_blocks0_maxFr_maxNr_seed1"
#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_inputs1_maxFr_maxNr_seed1"
#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_blocks1_maxFr_maxNr_seed1"
#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_inputs2_maxFr_maxNr_seed1"
#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_blocks2_maxFr_maxNr_seed1"
#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_hidden_maxFr_maxNr_seed1"
#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_recurrentout_maxFr_maxNr_seed1"
#PREFIX = "fnn07_act_i3_n2000_SCL0_7_TL37_position_maxFr_maxNr_seed1"
#PREFIX = "fnn07_seed2"
#PREFIX = "Retina"
PREFIX = "V1"

#PREFIX = "flyvis_Retina_i3_n400_model000"
#PREFIX = "flyvis_Lamina_i3_n350_model000"
#PREFIX = "flyvis_Medulla_i3_n550_model000"
#PREFIX = "flyvis_T_Tm_i3_n1700_model000"

# Per-layer parameters read from the actual mat file names in data/decompositions/.
# R      = number of tensor factors (F value in filename)
# method = normalisation method used when the mat file was created
# low_sf = whether the tensor was sliced to the low-SF stimulus set
_LAYER_PARAMS = {
    "fnn07_act_i3_n2000_SCL0_7_TL37_inputs0_maxFr_maxNr_seed1":      dict(R=13, method='relNorm', low_sf=True),
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks0_maxFr_maxNr_seed1":      dict(R=10, method='relNorm', low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_inputs1_maxFr_maxNr_seed1":      dict(R=11, method='relNorm', low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks1_maxFr_maxNr_seed1":      dict(R=7,  method='relNorm', low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_inputs2_maxFr_maxNr_seed1":      dict(R=19, method='relNorm', low_sf=True),
    "fnn07_act_i3_n2000_SCL0_7_TL37_blocks2_maxFr_maxNr_seed1":      dict(R=5,  method='relNorm', low_sf=True),
    "fnn07_act_i3_n2000_SCL0_7_TL37_hidden_maxFr_maxNr_seed1":       dict(R=10, method='Norm',    low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_recurrentout_maxFr_maxNr_seed1": dict(R=12, method='relNorm', low_sf=False),
    "fnn07_act_i3_n2000_SCL0_7_TL37_position_maxFr_maxNr_seed1":     dict(R=17, method='relNorm', low_sf=False),
    "fnn07_seed2": dict(R=14, method='relNorm', low_sf=False),
    "Retina":      dict(R=14,  method='relNorm', low_sf=True),
    "V1":          dict(R=17,  method='relNorm', low_sf=True),
    "flyvis_Retina_i3_n400_model000": dict(R=10, method='relNorm', low_sf=True),
    "flyvis_Lamina_i3_n350_model000": dict(R=10, method='relNorm', low_sf=True),
    "flyvis_Medulla_i3_n550_model000": dict(R=10, method='relNorm', low_sf=True),
    "flyvis_T_Tm_i3_n1700_model000": dict(R=10, method='relNorm', low_sf=True),
}

if PREFIX in _LAYER_PARAMS:
    _p     = _LAYER_PARAMS[PREFIX]
    R      = _p['R']
    method = _p['method']
    low_sf = _p['low_sf']
else:
    raise ValueError(f'Unknown PREFIX {PREFIX!r}. Add it to _LAYER_PARAMS above.')

# Minimum cumulative explained variance ratio for adaptive nPCs selection.
# nPCs is computed automatically after fitting PCA in Section 1.
MIN_EXPL_VAR_RATIO = 0.8

# Stimulus and direction counts (after low_sf filtering)
N_STIM = 6
N_DIR  = 8

# Colors and labels for stimuli (length = N_STIM)
STIMULUS_COLORS = [
    '#0070C0', '#00B0F0', '#00B050', '#92D050', '#FF0000', '#FFC000',
]
LABELS = [
    'LF-grat', 'HF-grat', '-1dot', '-3dot', '+1dot', '+3dot',
]

# Palette for encoding manifold (feature-map / cluster coloring)
PALETTE = np.array([
    '#03579b', '#0488d1', '#03a9f4', '#4fc3f7', '#b3e5fc',
    '#19237e', '#303f9f', '#3f51b5', '#7986cb', '#c5cae9',
    '#4a198c', '#7b21a2', '#9c27b0', '#ba68c8', '#e1bee7',
    '#88144f', '#c21f5b', '#e92663', '#f06292', '#f8bbd0',
    '#bf360c', '#e64a18', '#ff5722', '#ff8a65', '#ffccbc',
    '#f67f17', '#fbc02c', '#ffec3a', '#fff177', '#fdf9c3',
    '#33691d', '#689f38', '#8bc34a', '#aed581', '#ddedc8',
    '#253137', '#455a64', '#607d8b', '#90a4ae', '#cfd8dc',
])

optSF      = True
smooth_sig = 3

basedir_data = '../data/sampled'
basedir_mat  = '../data/decompositions'
basedir_wG   = '../data/graphs'
basedir_fig  = '../fig'

print(f'PREFIX={PREFIX}  R={R}  method={method!r}  low_sf={low_sf}')

## Section 1: Load Data <a id='section-1'></a>

Mirrors encoding_manifolds.ipynb cells 5–36. Run once, then collapse.

In [ ]:
# ---- Load raw tensor and neuron metadata --------------------------------

tensor4d = np.load(f'{basedir_data}/tensor4d_{PREFIX}.npy')
tensor4d = tensor4d - np.min(tensor4d)
neurons_used = np.load(f'{basedir_data}/neurons_used_{PREFIX}.npy')

# Select spatial frequencies
if low_sf:
    tensor4d = tensor4d[:, :6]
else:
    tensor4d = tensor4d[:, np.array([0, 6, 7, 8, 9, 10])]

print(f'tensor4d shape (N, S, D, T): {tensor4d.shape}')

In [ ]:
# ---- Process tensor data (mirrors encoding_manifolds.ipynb cell 7) ------
tensorX, relFRs, optStims = process_tensor_data(tensor4d, optSF, smooth_sig, method)

N, NSTIMS, NDIRS, TRIAL_LEN = tensor4d.shape
PREFIX2 = f'{method}_sig{smooth_sig}_n{N}'
if optSF and PREFIX not in ('Retina', 'V1'):
    PREFIX2 += '_SF'

CPMETHOD = 'shift'
AREA = 'deepnet' #if PREFIX not in ('Retina', 'V1') else PREFIX.lower()
tensorname = f'{PREFIX}-{PREFIX2}-{AREA}-{CPMETHOD}'
print('tensorname:', tensorname)

In [ ]:
# ---- Load precomputed tensor decomposition (mirrors cells 12, 18) -------
preComputed, Fs = loadPreComputedCP(tensorname, basedir_mat, specificFs=[R], verbose=1)
print(preComputed.keys())
rep, error = min(preComputed[R]['all_objs'].items(), key=lambda x: x[1])
print(f'Best rep #{rep} (error = {error:.3f})')

best_factors = preComputed[R]['all_factors'][rep]
best_lambdas = preComputed[R]['all_lambdas'][rep]

posnorms    = ~np.isclose(best_lambdas, 0)
lambdas     = best_lambdas[posnorms]
factors     = [f[:, posnorms] / np.linalg.norm(f[:, posnorms], axis=0, keepdims=1)
               for f in best_factors]

In [ ]:
# ---- Build neural matrix X (mirrors cells 24) ---------------------------
sigT  = tensorX
permT, fitT = getPermutedTensor(factors, lambdas, sigT, NDIRS)
scld_permT  = permT

X_raw, _ = getNeuralMatrix(
    scld_permT, factors, lambdas, NDIRS,
    all_zeroed_stims=None, order='F', verbose=False)

# PCA — fit on all R components first to determine adaptive nPCs
pca_neural = PCA(len(lambdas))
pcaX       = pca_neural.fit_transform(X_raw)  # (N_all, R)

# Adaptive nPCs: smallest number of PCs that explain >= MIN_EXPL_VAR_RATIO variance
nPCs = np.flatnonzero(np.cumsum(pca_neural.explained_variance_ratio_) > MIN_EXPL_VAR_RATIO)[0] + 1
print(f'{nPCs=}  (explains >{MIN_EXPL_VAR_RATIO:.0%} cumulative variance)')

plt.plot(pca_neural.explained_variance_ratio_, 'bo-')
plt.axvline(nPCs - 1, color='red', linestyle='--', label=f'nPCs={nPCs}')
plt.xlabel('PC'); plt.ylabel('Explained variance ratio')
plt.legend(); plt.title('Neural matrix PCA'); plt.show()

X = pcaX[:, :nPCs]   # (N_all, nPCs)
print(f'Neural matrix X shape: {X.shape}')

In [ ]:
# ---- Manual outlier removal (mirrors cells 26–27) -----------------------
# Adjust the slices below to match your encoding_manifolds.ipynb analysis.
D2_all = pwdists(X, sqdists=True)
N_all  = D2_all.shape[0]
D1_all = np.sqrt(D2_all)
mindists = np.min(D1_all + np.eye(N_all) * D1_all.max(), axis=0)

outls_far   = np.argsort(mindists)[::-1][:2]    # 3 most distant points
outls_close = np.argsort(mindists)[:5]          # 37 near-duplicate points

outliers_list = np.unique(np.append(outls_far, outls_close))
nonoutliers   = np.array([i for i in range(X.shape[0]) if i not in outliers_list])

# X_full: the PCA-reduced, outlier-filtered neural matrix — KEY INPUT
X_full = X[nonoutliers]   # shape (N_nonout, nPCs)
D2     = pwdists(X_full, sqdists=True)

print(f'X_full shape (N_nonout, nPCs): {X_full.shape}')
print(f'nonoutliers count: {len(nonoutliers)}')

In [ ]:
# ── IAN graph — loads from .npz cache if available and consistent ─────────
# Cache key includes PREFIX and R so each layer/R combo is stored separately.
# The notebook shares the same cache as 03_encoding_manifolds, so we check that
# the cached wG size matches the current nonoutliers before trusting it.
wG_path = f'{basedir_wG}/{PREFIX}_{R}.npz'

_use_cache = False
if os.path.exists(wG_path):
    _cache = np.load(wG_path, allow_pickle=False)
    if _cache['wG'].shape[0] == len(nonoutliers):
        wG_dense  = _cache['wG']
        wG        = csr_matrix(wG_dense)
        G         = (wG_dense > 0).astype(float)
        optScales = _cache['optScales']
        disc_pts  = [[i] for i in _cache['disc_pts_indices']]
        _use_cache = True
        print(f'Loaded cached IAN from {wG_path}')
    else:
        print(f'Cache size mismatch: wG has {_cache["wG"].shape[0]} nodes '
              f'but nonoutliers has {len(nonoutliers)} entries. Recomputing...')

if not _use_cache:
    print('Running IAN (this may take several minutes)...')
    solver = None  # set to 'GUROBI' for faster convergence
    G, wG, optScales, disc_pts = IAN('exact-precomputed-sq', D2, solver=solver)
    os.makedirs(basedir_wG, exist_ok=True)
    _disc_indices = np.array([disc_pts[di][0] for di in range(len(disc_pts))], dtype=np.int64)
    np.savez(wG_path, wG=wG.toarray(), optScales=optScales, disc_pts_indices=_disc_indices)
    print(f'IAN computed and cached → {wG_path}')

    # Handle any disconnected points from IAN
    wG, G, outliers_list, nonoutliers, _ = handle_disconnected_points(
        disc_pts, optScales, G, D2, outliers_list, neurons_used, X)
    X_full = X[nonoutliers]
    print(f'After IAN cleanup: {len(nonoutliers)} nonoutliers')

print(f'wG shape: {wG.shape}')

In [ ]:
# ---- Diffusion maps + MDS embedding (mirrors cells 34–36) ---------------
diffmap_y, diffmap_evals = diffusionMapSparseK(wG, 20, 1, t=1)
embedding_ = compute_mds_embedding(diffmap_y, nPCs, n_components=10)

# Pre-filter tensor to nonoutlier space so all subpop indexing is direct.
# Must come AFTER the IAN cell since that cell may update nonoutliers from cache.
tensor4d_nonout = tensor4d[nonoutliers]   # (N_nonout, S, D, T)

print(f'diffmap_y shape:  {diffmap_y.shape}')        # (N_nonout, 19)
print(f'embedding_ shape: {embedding_.shape}')       # (N_nonout, 10)
print(f'tensor4d_nonout:  {tensor4d_nonout.shape}')  # (N_nonout, S, D, T)

In [ ]:
# ---- HDBSCAN clustering (for Method 2 cluster selection) ----------------
cluster_labels, num_clusters, cond_tree, leaves = run_hdbscan_clustering(
    diffmap_y, nPCs, G, min_cluster_size=10)

print(f'Number of clusters: {num_clusters}')
print(f'Cluster label range: {cluster_labels.min()} to {cluster_labels.max()}')

# Quick 3D embedding overview colored by cluster
fig_overview, ax_overview = plot_encoding_manifold_3d(
    embedding_, cluster_labels, PALETTE, dcs=(0, 1, 2),
    title=f'Full population — {PREFIX}')
plt.show()

## Section 2: Choose Subpopulation <a id='section-2'></a>

Run **one** of the subsections below (Method 1, 2a, 2b, or 2c), then run the
validation cell at the end of this section.

### Method 1 — Dynamic Filtering

In [ ]:
# Compute per-neuron dynamic metrics over the full population (all N neurons).
# We remap to nonoutlier space in the filter cell below.
metrics = compute_dynamic_metrics(tensor4d, target_stim_idx=None)

# Inspect distributions (no selection yet)
fig_metrics, _ = plot_metric_distributions(metrics)
plt.show()

for k, v in metrics.items():
    print(f'{k:20s}: min={v.min():.4f}  max={v.max():.4f}  mean={v.mean():.4f}')

In [ ]:
# --- Config ---
METRIC     = 'speed'   # 'speed' | 'stability' | 'curvature' | 'classifiability' | 'pc_contrib'
PERCENTILE = 5       # percent of neurons to select (e.g. 2 → top or bottom 2%)
SIDE       = 'low'     # 'high' → top PERCENTILE%,  'low' → bottom PERCENTILE%

# --- Apply filter ---
if SIDE == 'high':
    selected_mask = filter_neurons_by_metric(metrics, METRIC, percentile_gt=100 - PERCENTILE)
else:
    selected_mask = filter_neurons_by_metric(metrics, METRIC, percentile_lt=PERCENTILE)

# Example compound filter (uncomment to use):
# selected_mask = (
#     filter_neurons_by_metric(metrics, 'speed', percentile_gt=99.75) &
#     filter_neurons_by_metric(metrics, 'stability', percentile_lt=50)
# )

# selected_mask is in full-population space (N neurons).
# Restrict to the nonoutlier set so subpop_indices align with tensor4d_nonout.
subpop_mask    = selected_mask[nonoutliers]   # nonoutlier-space bool, length N_nonout
subpop_indices = np.where(subpop_mask)[0]     # indices into tensor4d_nonout rows

print(f'Selected {selected_mask.sum()} / {len(selected_mask)} neurons (full space)')
print(f'After nonoutlier restriction: {subpop_mask.sum()} neurons')

# Overlay selected neurons on metric distributions (full-space mask for correct histograms)
fig_metrics2, _ = plot_metric_distributions(metrics, selected_mask)
plt.show()

In [ ]:
print(subpop_indices)

### Method 2a — Cluster Selection

In [ ]:
"""# Inspect cluster plot above, then set target cluster(s)
target_cluster = 0   # change after inspecting the overview plot

subpop_mask    = select_neurons_by_cluster(cluster_labels, target_cluster)
subpop_indices = np.where(subpop_mask)[0]

print(f'Cluster {target_cluster}: {subpop_mask.sum()} neurons selected')"""

### Method 2b — Bounding Box

In [ ]:
"""# Define bounding box in embedding space (dim index → (min, max))
bounds = {0: (-0.2, 0.2), 1: (0.1, 0.4)}
#bounds = {0: (0.2, 5), }

subpop_mask    = select_neurons_by_bbox(embedding_, bounds)
subpop_indices = np.where(subpop_mask)[0]

print(f'Bounding box selection: {subpop_mask.sum()} neurons')"""

### Method 2c — Seed + Radius

In [ ]:
"""seed_idx = 42    # index in nonoutlier space
radius   = 0.03

subpop_mask    = select_neurons_by_radius(embedding_, seed_idx, radius)
subpop_indices = np.where(subpop_mask)[0]

print(f'Radius selection (seed={seed_idx}, r={radius}): {subpop_mask.sum()} neurons')"""

### Validation — Where does the subpopulation sit?

In [ ]:
# 3D scatter: rest in gray, selected highlighted
fig_loc, ax_loc = plot_neuron_locations_in_manifold(
    embedding_, subpop_mask, dcs=(0, 1, 2))
ax_loc.set_title(f'Selected {subpop_mask.sum()} neurons in full embedding')
plt.show()

print(f'Subpopulation size in nonoutlier space: {len(subpop_indices)}')

## Section 3: Decoding Analysis <a id='section-3'></a>

In [ ]:
# subpop_indices are in nonoutlier space (0..N_nonout-1).
# tensor4d_nonout is already filtered to nonoutlier space, so index directly.
print(subpop_indices)
tensor4d_sub = tensor4d_nonout[subpop_indices]   # (K, S, D, T)

print(f'tensor4d_sub shape: {tensor4d_sub.shape}')

In [ ]:
# ---- Decoding manifolds: full population vs subpop (side by side) -------
coords_full, _ = compute_decoding_manifold(tensor4d_nonout, n_components=3)
coords_sub,  _ = compute_decoding_manifold(tensor4d_sub,    n_components=3)

lbls = np.array([lb for lb in LABELS for _ in range(N_DIR)])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), subplot_kw={'projection': '3d'})
for ax, coords, title in [
        (axes[0], coords_full, f'Full population ({len(nonoutliers)} neurons)'),
        (axes[1], coords_sub,  f'Subpop ({len(subpop_indices)} neurons)')]:
    for i, label in enumerate(LABELS):
        mask = lbls == label
        ax.scatter(coords[mask, 0], coords[mask, 1], coords[mask, 2],
                   color=STIMULUS_COLORS[i], s=100, edgecolors='none',
                   depthshade=False, label=label)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.grid(False)
    ax.view_init(elev=30, azim=45)
    ax.set_title(title, fontsize=10)

axes[0].legend(fontsize=7, loc='upper left', bbox_to_anchor=(-0.1, 1.0))
fig.suptitle(f'Decoding Manifold — {PREFIX}', fontsize=11)
fig.tight_layout()
plt.savefig(f'{basedir_fig}/decoding_manifolds/{PREFIX}_subpop_comparison.pdf',
            format='pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ---- Decoding trajectories: full population vs subpop (side by side) ----
trajs_full, _ = compute_decoding_trajectories(tensor4d_nonout, n_components=3)
trajs_sub,  _ = compute_decoding_trajectories(tensor4d_sub,    n_components=3)

def _plot_trajs_on_ax(ax, trajs, title):
    for i, traj in enumerate(trajs):
        color = STIMULUS_COLORS[(i // N_DIR) % len(STIMULUS_COLORS)]
        x, y, z = traj[:, 0], traj[:, 1], traj[:, 2] * 0.5
        ax.plot(x, y, z, color=color, linewidth=1.5, alpha=0.35)
        ax.scatter(x[0],  y[0],  z[0],  color='black', s=15, depthshade=False, edgecolors='none')
        ax.scatter(x[-1], y[-1], z[-1], color=color,   s=60, depthshade=False, edgecolors='none')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.grid(False)
    ax.view_init(elev=30, azim=45)
    ax.set_title(title, fontsize=10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), subplot_kw={'projection': '3d'})
_plot_trajs_on_ax(axes[0], trajs_full, f'Full population ({len(nonoutliers)} neurons)')
_plot_trajs_on_ax(axes[1], trajs_sub,  f'Subpop ({len(subpop_indices)} neurons)')
fig.suptitle(f'Decoding Trajectories — {PREFIX}', fontsize=11)
fig.tight_layout()
plt.savefig(f'{basedir_fig}/decoding_trajectories/crossings_{PREFIX}_subpop_comparison.pdf',
            format='pdf', bbox_inches='tight', dpi=300)
plt.show()

## Section 4: Encoding Analysis <a id='section-4'></a>

Rebuilds IAN + diffusion maps + MDS on just the subpopulation, then shows
the full population encoding manifold (subpop highlighted) alongside the
rebuilt subpop-only manifold.

### Full rebuild — fresh manifold from subpopulation only (minutes)

This re-runs IAN + diffusion maps + MDS on just the subpopulation.
Best for subpopulations of ≥ ~50 neurons.

In [ ]:
print(f'Running IAN on {len(subpop_indices)} neurons...')

enc = build_encoding_manifold_for_subpop(
    X_full, subpop_indices, nPCs=nPCs,
    solver=None,              # set 'GUROBI' for faster convergence
    n_diffmap_components=20,
    n_mds_components=10
)

print(f"X_sub shape:      {enc['X_sub'].shape}")
print(f"nonoutliers:      {len(enc['nonoutliers'])} clean neurons")
print(f"embedding_ shape: {enc['embedding_'].shape}")

In [ ]:
# Cluster the rebuilt subpop encoding manifold
sub_labels, sub_n_clusters, _, _ = run_hdbscan_clustering(
    enc['diffmap_y'], nPCs, enc['G'], min_cluster_size=5)

print(f'Subpop clusters: {sub_n_clusters}')

# ---- Encoding manifolds: full pop (subpop highlighted) vs rebuilt subpop --
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': '3d'})

# Left: full population, subpop colored in orange
ax = axes[0]
ax.scatter(embedding_[~subpop_mask, 0], embedding_[~subpop_mask, 1], embedding_[~subpop_mask, 2],
           c='lightgray', s=10, alpha=0.4, edgecolors='none', depthshade=False, label='rest')
ax.scatter(embedding_[subpop_mask, 0], embedding_[subpop_mask, 1], embedding_[subpop_mask, 2],
           c='darkorange', s=30, alpha=0.9, edgecolors='none', depthshade=False, label='subpop')
ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
ax.grid(False); ax.set_box_aspect([1, 1, 1])
ax.view_init(elev=30, azim=45)
ax.set_title(f'Full population ({len(nonoutliers)} neurons)\nsubpop highlighted', fontsize=10)
ax.legend(fontsize=8, loc='upper left')

# Right: rebuilt subpop-only manifold, single color
ax = axes[1]
ax.scatter(enc['embedding_'][:, 0], enc['embedding_'][:, 1], enc['embedding_'][:, 2],
           c='darkorange', s=30, alpha=0.8, edgecolors='none', depthshade=False)
ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
ax.grid(False); ax.set_box_aspect([1, 1, 1])
ax.view_init(elev=30, azim=45)
ax.set_title(f'Subpop rebuilt ({len(enc["nonoutliers"])} neurons)', fontsize=10)

fig.suptitle(f'Encoding Manifold — {PREFIX}', fontsize=11)
fig.tight_layout()
plt.savefig(f'{basedir_fig}/3d_manifolds/{PREFIX}_subpop_encoding_comparison.pdf',
            format='pdf', bbox_inches='tight', dpi=300)
plt.show()

## Section 5: PSTH Heatmaps <a id='section-5'></a>

In [ ]:
# ---- PSTH heatmap sixtuplet averaged over subpopulation -----------------
# tensorX is full-population space; map subpop from nonoutlier → full space.
plot_permT = np.reshape(tensorX, (len(tensorX), N_STIM, -1, N_DIR))
neuron_idx_full = nonoutliers[subpop_indices]   # indices into full population

# Pre-compute all panels and find global color range
panels = []
for stim in range(N_STIM):
    pst_orig = np.mean(plot_permT[neuron_idx_full, stim, :, :], axis=0).T  # (N_DIR, T)
    pst = np.zeros((pst_orig.shape[0], pst_orig.shape[1] + 5))
    pst[:, 5:] = pst_orig
    opt_dir = pst.mean(1).argmax()
    pst = np.roll(pst, (2 - opt_dir) % N_DIR, 0)
    panels.append(pst)

vmin = min(p.min() for p in panels)
vmax = max(p.max() for p in panels)

fig, axs = plt.subplots(3, 2, figsize=(5, 6))
for stim, pst in enumerate(panels):
    axs[stim // 2, stim % 2].imshow(pst, aspect='auto', interpolation='quadric',
                                     cmap='hot', vmin=vmin, vmax=vmax)
    axs[stim // 2, stim % 2].xaxis.set_visible(False)
    axs[stim // 2, stim % 2].yaxis.set_visible(False)
    axs[stim // 2, stim % 2].set_title(LABELS[stim], fontsize=8)

fig.suptitle(f'PSTHs — {PREFIX} subpop ({len(subpop_indices)} neurons)', fontsize=10)
plt.tight_layout()
plt.savefig(f'{basedir_fig}/{PREFIX}_subpop_psth.pdf', format='pdf', bbox_inches='tight', dpi=300)
plt.show()

## Section 6: Population-Fraction Sweep <a id='section-6'></a>

Systematically varies the fraction of neurons kept (100% → ~2%) under several
filtering strategies and measures how well full-population decoding is preserved.

**Metrics:**
- **k-NN accuracy** — can stimuli still be classified from the sub-population manifold?
- **Procrustes R²** — geometric similarity to the full-population manifold (invariant to rotation/scale)

**Strategies:**
- `random` — random subset (mean ± 1 std over 10 seeds)
- `speed  / [lo]` — highest / lowest early-transient speed
- `classif. ` — highest stimulus discriminability
- `stability ` — highest late-transient stability
- `pc_contrib ` — highest first-PC loading

> **Prerequisites:** Sections 1–3 must have been run so that `tensor4d_nonout`,
> `metrics`, and `decoding_coords_full` (= `coords_full`) are available.

In [ ]:
from src.subpop_utils import (select_top_k_by_metric, knn_decoding_accuracy,
                               procrustes_r2, variance_reproduced)

# ── Section 6 config ────────────────────────────────────────────────────────
# Large-end fractions; single-neuron fractions are prepended dynamically in
# the sweep cell once N_nonout is known.
FRACTIONS = np.array([0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40,
                      0.50, 0.60, 0.70, 0.80, 0.90, 1.00])
N_RANDOM_SEEDS = 10
rng = np.random.default_rng(0)

# metrics was computed in Section 2 (Method 1 cell).
# If you skipped Method 1, uncomment the line below:
# metrics = compute_dynamic_metrics(tensor4d, target_stim_idx=None)

print(f'N_nonout = {tensor4d_nonout.shape[0]}  |  base fractions: {FRACTIONS}')

In [ ]:
STRATEGIES = {
    'random':       {'type': 'random'},
    'speed ':      {'type': 'metric', 'metric': 'speed',           'high': True},
    'speed [lo]':      {'type': 'metric', 'metric': 'speed',           'high': False},
    'classif. ':   {'type': 'metric', 'metric': 'classifiability', 'high': True},
    'stability ':  {'type': 'metric', 'metric': 'stability',       'high': True},
    'pc_contrib ': {'type': 'metric', 'metric': 'pc_contrib',      'high': True},
}

# NOTE: metrics keys are in *full-population* space (N neurons).
# select_top_k_by_metric must therefore receive the full-space metrics dict,
# and the returned indices must be mapped to nonoutlier space before indexing
# tensor4d_nonout.  We build a helper mapping here once.
_full_to_nonout = {full_idx: no_idx
                   for no_idx, full_idx in enumerate(nonoutliers)}

def _nonout_indices_from_full(full_idx_arr):
    """Map full-population indices → nonoutlier-space indices (drop outliers)."""
    return np.array([_full_to_nonout[i] for i in full_idx_arr
                     if i in _full_to_nonout])

print('Strategies defined:', list(STRATEGIES))

In [ ]:
import time

N_nonout = tensor4d_nonout.shape[0]
stim_labels_full = np.repeat(np.arange(NSTIMS), NDIRS)   # (S*D,)

# Extend fractions down to single-neuron resolution
_single_frac = np.array([1, 2, 3, 5, 8, 13, 20]) / N_nonout
FRACTIONS_SWEEP = np.unique(np.concatenate([_single_frac, FRACTIONS]))

# Full-population reference manifold (already computed in Section 3 as coords_full)
decoding_coords_full = coords_full   # shape (S*D, 3)

results = {name: {'acc': [], 'r2': [], 'var': []} for name in STRATEGIES}

t0 = time.time()
for name, cfg in STRATEGIES.items():
    print(f'\n[{name}]')
    for f in FRACTIONS_SWEEP:
        k = max(1, int(round(f * N_nonout)))
        # Use as many PCA components as neurons allow (max 3)
        n_comp = min(3, k)

        if cfg['type'] == 'random':
            accs, r2s, vars_ = [], [], []
            for seed in range(N_RANDOM_SEEDS):
                idx = rng.choice(N_nonout, k, replace=False)
                coords_s, _ = compute_decoding_manifold(
                    tensor4d_nonout[idx], n_components=n_comp)
                accs.append(knn_decoding_accuracy(coords_s, stim_labels_full))
                r2s.append(procrustes_r2(decoding_coords_full, coords_s))
                vars_.append(variance_reproduced(decoding_coords_full, coords_s))
            results[name]['acc'].append((np.nanmean(accs), np.nanstd(accs)))
            results[name]['r2'].append((np.nanmean(r2s),   np.nanstd(r2s)))
            results[name]['var'].append((np.nanmean(vars_), np.nanstd(vars_)))

        else:
            full_idx = select_top_k_by_metric(
                metrics, cfg['metric'], k=int(round(f * len(nonoutliers))),
                high=cfg['high'])
            idx = _nonout_indices_from_full(full_idx)
            if len(idx) < 1:
                results[name]['acc'].append((np.nan, 0.0))
                results[name]['r2'].append((np.nan, 0.0))
                results[name]['var'].append((np.nan, 0.0))
                continue
            n_comp_s = min(3, len(idx))
            coords_s, _ = compute_decoding_manifold(
                tensor4d_nonout[idx], n_components=n_comp_s)
            results[name]['acc'].append(
                (knn_decoding_accuracy(coords_s, stim_labels_full), 0.0))
            results[name]['r2'].append(
                (procrustes_r2(decoding_coords_full, coords_s), 0.0))
            results[name]['var'].append(
                (variance_reproduced(decoding_coords_full, coords_s), 0.0))

        print(f'  f={f:.4f}  k={k}'
              f'  acc={results[name]["acc"][-1][0]:.3f}'
              f'  r2={results[name]["r2"][-1][0]:.3f}'
              f'  var={results[name]["var"][-1][0]:.3f}')

print(f'\nTotal sweep time: {time.time()-t0:.1f} s')

In [ ]:
import os, matplotlib.pyplot as plt
from tueplots import bundles
plt.rcParams.update(bundles.neurips2021(usetex=False))

_COLORS = {
    'random':       '#607d8b',
    'speed ':      '#e64a18',
    'speed [lo]':      '#fbc02c',
    'classif. ':   '#0488d1',
    'stability ':  '#00b050',
    'pc_contrib ': '#9c27b0',
}

fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharey=False)
ax_acc, ax_r2, ax_var = axes

panels = [
    (ax_acc, 'acc', 'k-NN accuracy',     'Decoding accuracy'),
    (ax_r2,  'r2',  'Procrustes R²',     'Geometric preservation'),
    (ax_var, 'var', 'Variance reproduced', 'Variance reproduced'),
]

for name, res in results.items():
    color = _COLORS.get(name, 'gray')
    ls = '--' if name == 'random' else '-'
    lw = 1.2

    for ax, key, _, _ in panels:
        vals    = np.array([v[0] for v in res[key]])
        std     = np.array([v[1] for v in res[key]])
        ax.plot(FRACTIONS_SWEEP, vals, color=color, lw=lw, ls=ls, label=name)
        if name == 'random':
            ax.fill_between(FRACTIONS_SWEEP, vals - std, vals + std,
                            color=color, alpha=0.2)

for ax, _, ylabel, title in panels:
    ax.set_xlabel('Fraction of neurons selected')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=9)
    ax.set_xlim(1.0, FRACTIONS_SWEEP[0])   # inverted: full pop on left
    ax.tick_params(labelsize=7)

ax_acc.legend(fontsize=6.5, loc='lower left', ncol=2)

# Reference line at var=1 (subpop exactly reproduces full-pop variance)
ax_var.axhline(1.0, color='black', lw=0.6, ls=':', alpha=0.5)

fig.suptitle(f'Population-fraction sweep — {PREFIX}', fontsize=9)
fig.tight_layout()

os.makedirs(f'{basedir_fig}/subpop', exist_ok=True)
fig.savefig(f'{basedir_fig}/subpop/{PREFIX}_fraction_sweep.pdf',
            format='pdf', bbox_inches='tight')
plt.show()
print('Saved to', f'{basedir_fig}/subpop/{PREFIX}_fraction_sweep.pdf')